In [0]:
# =============================================================
# lineage.py
# Layer     : Bronze
# Purpose   : Audit / lineage column injection only.
#             Adds metadata columns to every Bronze row so
#             data engineers can always trace where a row
#             came from and when it was loaded.
#
# STRICT RULE: No reads. No writes. No schemas.
#              Only column transformations.
# =============================================================

from pyspark.sql.functions import (
    current_timestamp,
    input_file_name,
    lit,
    to_timestamp,
    col
)


from pyspark.sql.functions import regexp_extract, to_date # Add to_date if you want a DateType

def add_lineage_columns(df, table_name: str, load_date: str, table_type: str) -> object:
    
    # Base lineage dataframe
    df_transformed = (
        df
        .withColumn("_bronze_load_timestamp", current_timestamp())
        .withColumn("_source_file", input_file_name())
        .withColumn("_table_name", lit(table_name))
        .withColumn("_table_type", lit(table_type))
        .withColumn("_is_quarantined", lit(False))
    )
    
    if table_type == "fact":
        # Extract yyyy-mm-dd format out of the 'year=XXXX/month=XX/day=XX' file path string
        df_transformed = df_transformed.withColumn(
            "_load_date", 
            regexp_extract(col("_source_file"), r"year=(\d{4})/month=(\d{2})/day=(\d{2})", 0)
            # Reformat to yyyy-mm-dd if path pattern is different, or use concat logic
        )
    else:
        # Fall back to the pipeline/widget execution date for dimensions
        df_transformed = df_transformed.withColumn("_load_date", lit(load_date))
        
    return df_transformed

def get_lineage_column_names() -> list:
    """Return list of all lineage column names — useful for Silver to exclude them."""
    return [
        "_bronze_load_timestamp",
        "_source_file",
        "_load_date",
        "_table_name",
        "_table_type",
        "_is_quarantined",
    ]

print("[lineage] Loaded. Lineage columns ready:", get_lineage_column_names())